# Predict Customer Churn
## Score: .91340

In [56]:
N_FOLDS = 5
N_SEEDS = 3
OPTUNA_TRIALS = 50
USE_RF_ET = True
USE_ORIGINAL_DATA = True
TARGET_ENC_M = 35

In [57]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])

if USE_ORIGINAL_DATA:
    original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = original.drop(columns=['customerID'])
    original = original[train.columns.drop('id')]
    train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
    sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
else:
    train_combined = train.drop(columns=['id'])
    sample_weights = np.ones(len(train_combined))

y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
print(f"Train: {len(X_full)}, Original: {USE_ORIGINAL_DATA}")

Train: 601237, Original: True


In [58]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

for col in ['Contract', 'PaymentMethod', 'InternetService']:
    X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=TARGET_ENC_M)
    X_full = X_full.drop(columns=[col])
    X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [59]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [60]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-05 22:09:31,029] A new study created in memory with name: no-name-0e8a6edf-f543-4744-9e23-fed0dc3b185a


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-05 22:10:14,445] Trial 0 finished with value: 0.9155750906228066 and parameters: {'n_estimators': 581, 'max_depth': 3, 'learning_rate': 0.13098956065267453, 'subsample': 0.7796611681572972, 'colsample_bytree': 0.8703885862015646, 'scale_pos_weight': 2.448315408621575, 'reg_alpha': 0.4742690336971851, 'reg_lambda': 7.8713612835021465, 'min_child_weight': 2}. Best is trial 0 with value: 0.9155750906228066.
[I 2026-03-05 22:11:16,624] Trial 1 finished with value: 0.915331372057666 and parameters: {'n_estimators': 543, 'max_depth': 7, 'learning_rate': 0.07524489423628145, 'subsample': 0.8901541178666411, 'colsample_bytree': 0.6439390291314597, 'scale_pos_weight': 1.1767819407666844, 'reg_alpha': 0.038268048780516435, 'reg_lambda': 0.005602066977423228, 'min_child_weight': 6}. Best is trial 0 with value: 0.9155750906228066.
[I 2026-03-05 22:12:28,104] Trial 2 finished with value: 0.9156964055381612 and parameters: {'n_estimators': 624, 'max_depth': 4, 'learning_rate': 0.072272766

In [61]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")

[I 2026-03-05 23:02:31,177] A new study created in memory with name: no-name-5d28f2d7-857c-40cd-b648-f6e5f40c8ae8


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-05 23:03:03,321] Trial 0 finished with value: 0.9152961712610601 and parameters: {'n_estimators': 304, 'max_depth': 5, 'learning_rate': 0.08116779754314606, 'subsample': 0.8923902569514841, 'colsample_bytree': 0.9036045233033614, 'scale_pos_weight': 2.641322614870659, 'reg_alpha': 1.2594726231630544, 'reg_lambda': 0.20318930237297453, 'min_child_samples': 23}. Best is trial 0 with value: 0.9152961712610601.
[I 2026-03-05 23:03:42,670] Trial 1 finished with value: 0.9151332144446226 and parameters: {'n_estimators': 435, 'max_depth': 8, 'learning_rate': 0.13992077205696826, 'subsample': 0.8035867516858596, 'colsample_bytree': 0.7457032714089362, 'scale_pos_weight': 2.51681061131794, 'reg_alpha': 0.007797544870886802, 'reg_lambda': 0.7450538106201536, 'min_child_samples': 33}. Best is trial 0 with value: 0.9152961712610601.
[I 2026-03-05 23:04:09,159] Trial 2 finished with value: 0.9146296658415952 and parameters: {'n_estimators': 449, 'max_depth': 3, 'learning_rate': 0.0588202

In [62]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-05 23:46:13,037] A new study created in memory with name: no-name-2fb5cad4-d7b3-49c1-8acf-c3163cec163f


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-05 23:48:58,825] Trial 0 finished with value: 0.915501591216054 and parameters: {'iterations': 770, 'depth': 5, 'learning_rate': 0.07236430048855572, 'subsample': 0.7466616021076194, 'colsample_bylevel': 0.6429936723904944, 'scale_pos_weight': 2.647558172195131, 'l2_leaf_reg': 1.5951887646053569}. Best is trial 0 with value: 0.915501591216054.
[I 2026-03-05 23:50:41,965] Trial 1 finished with value: 0.9154381119649493 and parameters: {'iterations': 429, 'depth': 6, 'learning_rate': 0.10266207475425644, 'subsample': 0.8579750028349828, 'colsample_bylevel': 0.7179466101948611, 'scale_pos_weight': 1.3355589262472067, 'l2_leaf_reg': 0.26242637557592025}. Best is trial 0 with value: 0.915501591216054.
[I 2026-03-05 23:52:20,534] Trial 2 finished with value: 0.9149752736214621 and parameters: {'iterations': 617, 'depth': 3, 'learning_rate': 0.1019658467932943, 'subsample': 0.7339743506761216, 'colsample_bylevel': 0.8613254227007712, 'scale_pos_weight': 3.1806525021437415, 'l2_leaf

In [63]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

n_models = 3 if not USE_RF_ET else 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))

def get_models(seed, fold):
    models = [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1),
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
    ]
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
    return models

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

def neg_auc(w):
    blend = oof @ w
    return -roc_auc_score(y_full, blend)

best_auc = -1
best_w = None
for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.2]*5)]:
    result = minimize(neg_auc, x0=x0, method='SLSQP',
                      bounds=[(0, 1)]*n_models,
                      constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    auc = -result.fun
    if auc > best_auc:
        best_auc = auc
        best_w = result.x
blend_weights = best_w
blend_oof = oof @ blend_weights
names = ['XGB','LGB','Cat'] + (['RF','ET'] if USE_RF_ET else [])
print(f"Blend OOF AUC: {roc_auc_score(y_full, blend_oof):.4f}")
print(f"Weights: {dict(zip(names, np.round(blend_weights, 4)))}")

Blend OOF AUC: 0.9153
Weights: {'XGB': np.float64(0.2), 'LGB': np.float64(0.2), 'Cat': np.float64(0.2), 'RF': np.float64(0.2), 'ET': np.float64(0.2)}


In [64]:
all_test_preds = [test_preds @ blend_weights]
for base_seed in [123, 456, 789, 2024, 2025][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(tp @ blend_weights)

final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.148176
1,594195,0.001960
2,594196,0.223736
3,594197,0.007854
4,594198,0.695177
